In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Helper to produce required aggregates for a telemetry dataframe
def summarize_telemetry(df, name_col):
    # distinct cmdb_id count and top 20 cmdb_id values (by frequency)
    cmdb_count = df['cmdb_id'].nunique()
    top20_cmdb = df['cmdb_id'].value_counts().head(20).reset_index()
    top20_cmdb.columns = ['cmdb_id', 'count']

    # top 20 distinct KPI/trace/log names with occurrence counts
    top20_names = df[name_col].value_counts().head(20).reset_index()
    top20_names.columns = [name_col, 'count']

    # Group by (cmdb_id, name_col) and compute count, min(ts), max(ts), and global P95 of 'value'
    # Compute aggregations on the entire series (no filtering)
    agg = df.groupby(['cmdb_id', name_col]).agg(
        count_points=('value', 'size'),
        min_ts_raw=('timestamp', 'min'),
        max_ts_raw=('timestamp', 'max'),
        p95_value=('value', lambda s: float(np.nanpercentile(s.values, 95)) if len(s) > 0 else np.nan)
    ).reset_index()

    # Convert raw min/max timestamps (Unix seconds) to UTC datetimes
    agg['min_ts'] = pd.to_datetime(agg['min_ts_raw'], unit='s', utc=True)
    agg['max_ts'] = pd.to_datetime(agg['max_ts_raw'], unit='s', utc=True)

    # Keep and order columns compactly
    agg = agg[['cmdb_id', name_col, 'count_points', 'min_ts', 'max_ts', 'p95_value']]
    agg = agg.sort_values('count_points', ascending=False).head(50)

    return cmdb_count, top20_cmdb, top20_names, agg

# Summarize each telemetry file
metric_cmdb_count, metric_top20_cmdb, metric_top20_kpis, metric_grouped_top = summarize_telemetry(metric, 'kpi_name')
trace_cmdb_count, trace_top20_cmdb, trace_top20_traces, trace_grouped_top = summarize_telemetry(trace, 'trace_name')
log_cmdb_count, log_top20_cmdb, log_top20_logs, log_grouped_top = summarize_telemetry(log, 'log_name')

# Summarize error_logs.csv
error_cmdb_count = error_logs['cmdb_id'].nunique()
error_top20_cmdb = error_logs['cmdb_id'].value_counts().head(20).reset_index()
error_top20_cmdb.columns = ['cmdb_id', 'count']
error_top20_messages = error_logs['message'].value_counts().head(20).reset_index()
error_top20_messages.columns = ['message', 'count']

# Incident window (given) as both Unix timestamps and ISO UTC strings
incident_start_iso = pd.Timestamp("2024-01-17 18:25:29", tz='UTC')
incident_end_iso = pd.Timestamp("2024-01-17 18:55:29", tz='UTC')
incident_window = pd.DataFrame([{
    'start_unix': int(incident_start_iso.timestamp()),
    'end_unix': int(incident_end_iso.timestamp()),
    'start_iso_utc': incident_start_iso.isoformat(),
    'end_iso_utc': incident_end_iso.isoformat()
}])

# Compact outputs (limited rows where applicable). Display by variable names.
metric_cmdb_count, metric_top20_cmdb, metric_top20_kpis, metric_grouped_top, \
trace_cmdb_count, trace_top20_cmdb, trace_top20_traces, trace_grouped_top, \
log_cmdb_count, log_top20_cmdb, log_top20_logs, log_grouped_top, \
error_cmdb_count, error_top20_cmdb, error_top20_messages, incident_window

```
Out[1]:
```
summary = (
    "Summary of telemetry analysis (compact):\n\n"
    "Incident window:\n"
    "- Unix: start=1705515929, end=1705517729\n"
    "- ISO UTC: 2024-01-17T18:25:29+00:00 to 2024-01-17T18:55:29+00:00\n\n"
    "metric.csv:\n"
    "- Distinct cmdb_id count: 12\n"
    "- Top cmdb_ids (by sample frequency, up to 12 shown): adservice, cartservice, checkoutservice, emailservice, paymentservice, frontend, recommendationservice, currencyservice, shippingservice, productcatalogservice, redis, frontend-external\n"
    "- Top KPI names (top 8 shown with counts): cpu (275), workload (275), socket (275), mem (275), latency-50 (250), latency-90 (250), diskio (175), error (36)\n"
    "- Grouped summary (cmdb_id, kpi_name): many groups have 25 points spanning ~2024-01-17T18:28:00Z to ~2024-01-17T18:52:00Z. Examples of global P95 (computed across full available series): adservice cpu p95 ≈ 0.7986, adservice diskio p95 ≈ 72487.88, productcatalogservice mem p95 ≈ 1.07e7. (Top 50 groups by point count were produced.)\n\n"
    "trace.csv:\n"
    "- Distinct cmdb_id count: 8\n"
    "- Top cmdb_ids (by sample frequency): checkoutservice, frontendservice, root, recommendationservice, productcatalogservice, currencyservice, emailservice, paymentservice\n"
    "- Top trace names (top 20 show many 'trace.from_root.*' and service-specific rows): trace.from_root.duration_mean, trace.from_root.duration_p95, trace.from_root.error_rate, trace.from_root.row_count, trace.from_checkoutservice.* , trace.from_frontendservice.* , trace.to_productcatalogservice.* , trace.to_checkoutservice.*\n"
    "- Grouped summary: many (cmdb_id, trace_name) groups have 25 points between ~18:28 and ~18:52 UTC. Example p95 values: checkoutservice trace.from_checkoutservice.row_count p95 = 292.6; checkoutservice trace.from_checkoutservice.duration_p95 ≈ 0.2036.\n\n"
    "log.csv:\n"
    "- Distinct cmdb_id count: 10\n"
    "- Top cmdb_ids: adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, paymentservice, recommendationservice, shippingservice, redis\n"
    "- Top log names: log.error_count (229), log.row_count (229)\n"
    "- Grouped summary: most (cmdb_id, log_name) groups have 25 points covering ~18:28 to ~18:52 UTC (redis has 4 points between ~18:32 and ~18:47). Examples: frontend log.row_count p95 ≈ 2795.6, currencyservice log.row_count p95 ≈ 2521.2.\n\n"
    "error_logs.csv:\n"
    "- Distinct cmdb_id count: 0 (no cmdb_id entries found)\n"
    "- Top cmdb_id values: none\n"
    "- Top messages: none (empty)\n\n"
    "Notes:\n"
    "- All min/max timestamps in grouped tables were parsed and shown in UTC. Global P95 values were computed across the full available series for each group before any time-window filtering.\n"
    "- If you want more detail for any specific cmdb_id or KPI/trace/log name (e.g., the top 50 groups or raw group rows), request which component/name and I will display a focused table (limited rows as requested)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(12,                   cmdb_id  count
0               adservice    175
1             cartservice    175
2         checkoutservice    175
3            emailservice    175
4          paymentservice    175
5                frontend    175
6   recommendationservice    175
7         currencyservice    150
8         shippingservice    150
9   productcatalogservice    150
10                  redis    100
11      frontend-external     36,      kpi_name  count
0         cpu    275
1    workload    275
2      socket    275
3         mem    275
4  latency-50    250
5  latency-90    250
6      diskio    175
7       error     36,                   cmdb_id    kpi_name  count_points                    min_ts                    max_ts     p95_value
0               adservice         cpu            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00  7.986226e-01
1               adservice      diskio            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00  7.248788e+04
2               adservice  latency-50            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00  3.016158e-03
3               adservice  latency-90            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00  4.633668e-03
4               adservice         mem            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00  1.097739e+08
..                    ...         ...           ...                       ...                       ...           ...
47         paymentservice         mem            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00  4.115919e+07
46         paymentservice  latency-90            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00  7.711233e-03
48         paymentservice      socket            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00  3.000000e+00
49         paymentservice    workload            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00  1.133600e+00
53  productcatalogservice         mem            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00  1.073471e+07

[50 rows x 6 columns], 8,                  cmdb_id  count
0        checkoutservice    800
1        frontendservice    700
2                   root    700
3  recommendationservice    500
4  productcatalogservice    400
5        currencyservice    300
6           emailservice    200
7         paymentservice    200,                                       trace_name  count
0                  trace.from_root.duration_mean    175
1                   trace.from_root.duration_p95    175
2                     trace.from_root.error_rate    175
3                      trace.from_root.row_count    175
4           trace.from_checkoutservice.row_count    125
5          trace.from_checkoutservice.error_rate    125
6        trace.from_checkoutservice.duration_p95    125
7       trace.from_checkoutservice.duration_mean    125
8           trace.from_frontendservice.row_count    125
9          trace.from_frontendservice.error_rate    125
10       trace.from_frontendservice.duration_p95    125
11      trace.from_frontendservice.duration_mean    125
12      trace.to_productcatalogservice.row_count    100
13     trace.to_productcatalogservice.error_rate    100
14   trace.to_productcatalogservice.duration_p95    100
15  trace.to_productcatalogservice.duration_mean    100
16        trace.to_checkoutservice.duration_mean     75
17         trace.to_checkoutservice.duration_p95     75
18           trace.to_checkoutservice.error_rate     75
19            trace.to_checkoutservice.row_count     75,             cmdb_id                                trace_name  count_points                    min_ts                    max_ts   p95_value
0   checkoutservice  trace.from_checkoutservice.duration_mean            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00    0.070861
1   checkoutservice   trace.from_checkoutservice.duration_p95            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00    0.203600
2   checkoutservice     trace.from_checkoutservice.error_rate            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00    0.000000
3   checkoutservice      trace.from_checkoutservice.row_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00  292.600000
4   checkoutservice  trace.from_frontendservice.duration_mean            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00    0.658550
..              ...                                       ...           ...                       ...                       ...         ...
45     emailservice   trace.from_checkoutservice.duration_p95            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00    0.000582
46     emailservice     trace.from_checkoutservice.error_rate            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00    0.000000
47     emailservice      trace.from_checkoutservice.row_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00   34.400000
48     emailservice             trace.from_root.duration_mean            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00    0.000052
49     emailservice              trace.from_root.duration_p95            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00    0.000092

[50 rows x 6 columns], 10,                  cmdb_id  count
0              adservice     50
1            cartservice     50
2        checkoutservice     50
3        currencyservice     50
4           emailservice     50
5               frontend     50
6         paymentservice     50
7  recommendationservice     50
8        shippingservice     50
9                  redis      8,           log_name  count
0  log.error_count    229
1    log.row_count    229,                   cmdb_id         log_name  count_points                    min_ts                    max_ts  p95_value
0               adservice  log.error_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00        0.0
1               adservice    log.row_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00      468.8
2             cartservice  log.error_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00        0.0
3             cartservice    log.row_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00      815.2
4         checkoutservice  log.error_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00        0.0
5         checkoutservice    log.row_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00      103.2
6         currencyservice  log.error_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00        0.0
7         currencyservice    log.row_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00     2521.2
8            emailservice  log.error_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00        0.0
9            emailservice    log.row_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00       34.4
10               frontend  log.error_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00        0.0
11               frontend    log.row_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00     2795.6
12         paymentservice  log.error_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00        0.0
13         paymentservice    log.row_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00       68.8
14  recommendationservice  log.error_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00        0.0
15  recommendationservice    log.row_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00      587.6
19        shippingservice    log.row_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00      475.6
18        shippingservice  log.error_count            25 2024-01-17 18:28:00+00:00 2024-01-17 18:52:00+00:00        0.0
17                  redis    log.row_count             4 2024-01-17 18:32:00+00:00 2024-01-17 18:47:00+00:00        5.0
16                  redis  log.error_count             4 2024-01-17 18:32:00+00:00 2024-01-17 18:47:00+00:00        0.0, 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [message, count]
Index: [],    start_unix    end_unix              start_iso_utc                end_iso_utc
0  1705515929  1705517729  2024-01-17T18:25:29+00:00  2024-01-17T18:55:29+00:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing 'metric' DataFrame loaded previously.
# Incident window (inclusive)
start_unix = 1705515929
end_unix = 1705517729

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric series (no filtering)
df_p95 = metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].agg(
    global_p95=lambda s: float(np.nanpercentile(s.values, 95)) if len(s) > 0 else np.nan
)

# 2) Filter metric rows to the incident window (inclusive)
window = metric[(metric['timestamp'] >= start_unix) & (metric['timestamp'] <= end_unix)].copy()

# Merge global_p95 into the windowed rows
window = window.merge(df_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Parse UTC timestamps for reporting
window['ts_utc'] = pd.to_datetime(window['timestamp'], unit='s', utc=True)

# 3) Mark anomaly where value >= global_p95 (global_p95 computed above)
# If global_p95 is NaN, mark anomaly as False
window['anomaly'] = window['value'] >= window['global_p95']
window['anomaly'] = window['anomaly'].fillna(False).astype(bool)

# 4) Aggregations per (cmdb_id, kpi_name)

# Basic counts and max in window
totals = window.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    total_points_in_window=('value', 'size'),
    anomaly_point_count_in_window=('anomaly', 'sum'),
    max_value_in_window=('value', 'max')
)

# Earliest and latest anomaly timestamps in UTC (for groups that have anomaly rows)
earliest_latest = (
    window[window['anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        earliest_anomaly_timestamp_utc=('ts_utc', 'min'),
        latest_anomaly_timestamp_utc=('ts_utc', 'max')
    )
)

# Longest consecutive anomaly streak in window per group
def longest_consecutive_streak(df_group):
    # ensure sorted by timestamp
    arr = df_group.sort_values('timestamp')['anomaly'].astype(bool).values
    max_run = 0
    cur = 0
    for val in arr:
        if val:
            cur += 1
            if cur > max_run:
                max_run = cur
        else:
            cur = 0
    return max_run

streaks = (
    window.groupby(['cmdb_id', 'kpi_name'])
    .apply(lambda g: longest_consecutive_streak(g))
    .reset_index(name='longest_consecutive_anomaly_streak_in_window')
)

# Assemble final summary: start from df_p95 to include all groups' global_p95,
# then left-merge windowed aggregates (groups with zero points in window will have NaNs -> fill)
summary = df_p95.merge(totals, on=['cmdb_id', 'kpi_name'], how='left') \
                .merge(earliest_latest, on=['cmdb_id', 'kpi_name'], how='left') \
                .merge(streaks, on=['cmdb_id', 'kpi_name'], how='left')

# Fill missing numeric aggregates for groups absent in window
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['anomaly_point_count_in_window'] = summary['anomaly_point_count_in_window'].fillna(0).astype(int)
summary['longest_consecutive_anomaly_streak_in_window'] = summary['longest_consecutive_anomaly_streak_in_window'].fillna(0).astype(int)

# Keep max_value_in_window as float (NaN if no points)
# earliest/latest anomaly timestamps remain NaT if no anomalies

# For sorting, compute severity delta = max_value_in_window - global_p95
# If max_value_in_window is NaN, set severity to -inf so they sort last
summary['severity_delta'] = summary['max_value_in_window'] - summary['global_p95']
summary['severity_delta'] = summary['severity_delta'].fillna(-np.inf)

# 5) Sort and limit to top 20 rows as requested
summary_sorted = summary.sort_values(
    by=['anomaly_point_count_in_window', 'longest_consecutive_anomaly_streak_in_window', 'severity_delta'],
    ascending=[False, False, False]
)

# Select and order required columns
final_cols = [
    'cmdb_id',
    'kpi_name',
    'global_p95',
    'total_points_in_window',
    'anomaly_point_count_in_window',
    'earliest_anomaly_timestamp_utc',
    'latest_anomaly_timestamp_utc',
    'max_value_in_window',
    'longest_consecutive_anomaly_streak_in_window'
]

summary_top20 = summary_sorted[final_cols].head(20)

# Display the compact summary table (at most 20 rows)
summary_top20

```
Out[2]:
```
Summary of anomalies in metric.csv for the incident window (computed p95 on each full series, then flagged window rows >= that p95):

Incident window
- Unix: start=1705515929, end=1705517729
- ISO UTC: 2024-01-17T18:25:29+00:00 to 2024-01-17T18:55:29+00:00

Method (brief)
- Global P95 was computed per (cmdb_id, kpi_name) using the entire available metric series before any window filtering.
- Window rows (inclusive) were flagged as anomalies when value >= that global P95.
- The table below lists the top groups by anomaly count (then by longest streak and severity delta).

Top anomalous (cmdb_id, kpi_name) groups (compact):
1. adservice — socket: 25/25 anomalies, earliest 2024-01-17T18:28:00Z, latest 2024-01-17T18:52:00Z, longest streak 25
2. cartservice — diskio: 25/25 anomalies, 2024-01-17T18:28:00Z → 18:52:00Z, streak 25
3. cartservice — socket: 25/25 anomalies, 18:28 → 18:52, streak 25
4. checkoutservice — diskio: 25/25 anomalies, 18:28 → 18:52, streak 25
5. checkoutservice — socket: 25/25 anomalies, 18:28 → 18:52, streak 25
6. currencyservice — socket: 25/25 anomalies, 18:28 → 18:52, streak 25
7. emailservice — diskio: 25/25 anomalies, 18:28 → 18:52, streak 25
8. emailservice — socket: 25/25 anomalies, 18:28 → 18:52, streak 25
9. frontend — error: 25/25 anomalies, 18:28 → 18:52, streak 25
10. paymentservice — diskio: 25/25 anomalies, 18:28 → 18:52, streak 25
11. paymentservice — socket: 25/25 anomalies, 18:28 → 18:52, streak 25
12. productcatalogservice — socket: 25/25 anomalies, 18:28 → 18:52, streak 25
13. recommendationservice — diskio: 25/25 anomalies, 18:28 → 18:52, streak 25
14. recommendationservice — socket: 25/25 anomalies, 18:28 → 18:52, streak 25
15. redis — socket: 25/25 anomalies, 18:28 → 18:52, streak 25
16. shippingservice — socket: 25/25 anomalies, 18:28 → 18:52, streak 25
17. frontend — socket: 12/25 anomalies, earliest anomaly 2024-01-17T18:41:00Z, latest 18:52:00Z, longest streak 12
18. frontend-external — error: 11/11 anomalies (11 points in-window total), earliest 2024-01-17T18:42:00Z → 18:52:00Z, streak 11
19. paymentservice — mem: 2/25 anomalies, earliest 2024-01-17T18:33:00Z → 18:34:00Z, streak 2 (max value in-window slightly above p95)
20. currencyservice — workload: 2/25 anomalies, earliest 2024-01-17T18:33:00Z → 18:34:00Z, streak 2

Key observations
- Many services show socket-related anomalies (socket metric) across the entire incident window (25/25 points flagged), with identical earliest/latest timestamps (18:28 → 18:52 UTC). This pattern appears across adservice, cartservice, checkoutservice, currencyservice, emailservice, paymentservice, productcatalogservice, recommendationservice, redis, shippingservice, etc.
- Disk I/O (diskio) anomalies also appear across many services for the full window (25/25).
- frontend socket anomalies start later in the window (from ~18:41 UTC).
- A few groups show only short anomaly bursts (payment mem and currency workload: 2 points around 18:33–18:34 UTC).
- For many flagged groups the max_value_in_window equals the computed global P95 (i.e., values in-window met or matched the threshold across the window). A few groups (e.g., paymentservice mem) show max > global P95 (small positive delta).

Recommended next steps
- Investigate widespread socket metric anomalies across many services simultaneously — suggests a common networking/socket-level issue or a metric normalization/configuration effect.
- Check disk I/O for the same services (diskio) since diskio also shows broad anomalies.
- Correlate with traces/logs and network infrastructure telemetry to determine if this is a real resource/error condition or a metric normalization/reporting artifact.

If you want, I can:
- Show the numeric global_p95 and max_value deltas for any of the listed groups,
- Or display the full top-50 grouped summary rows for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  kpi_name    global_p95  total_points_in_window  anomaly_point_count_in_window earliest_anomaly_timestamp_utc latest_anomaly_timestamp_utc  max_value_in_window  longest_consecutive_anomaly_streak_in_window
5               adservice    socket  4.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         4.000000e+00                                            25
8             cartservice    diskio  0.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         0.000000e+00                                            25
12            cartservice    socket  4.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         4.000000e+00                                            25
15        checkoutservice    diskio  0.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         0.000000e+00                                            25
19        checkoutservice    socket  9.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         9.000000e+00                                            25
25        currencyservice    socket  4.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         4.000000e+00                                            25
28           emailservice    diskio  0.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         0.000000e+00                                            25
32           emailservice    socket  3.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         3.000000e+00                                            25
35               frontend     error  0.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         0.000000e+00                                            25
44         paymentservice    diskio  0.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         0.000000e+00                                            25
48         paymentservice    socket  3.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         3.000000e+00                                            25
54  productcatalogservice    socket  4.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         4.000000e+00                                            25
57  recommendationservice    diskio  0.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         0.000000e+00                                            25
61  recommendationservice    socket  4.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         4.000000e+00                                            25
66                  redis    socket  4.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         4.000000e+00                                            25
71        shippingservice    socket  2.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         2.000000e+00                                            25
39               frontend    socket  3.500000e+01                      25                             12      2024-01-17 18:41:00+00:00    2024-01-17 18:52:00+00:00         3.500000e+01                                            12
41      frontend-external     error  0.000000e+00                      11                             11      2024-01-17 18:42:00+00:00    2024-01-17 18:52:00+00:00         0.000000e+00                                            11
47         paymentservice       mem  4.115919e+07                      25                              2      2024-01-17 18:33:00+00:00    2024-01-17 18:34:00+00:00         4.129922e+07                                             2
26        currencyservice  workload  8.284842e+01                      25                              2      2024-01-17 18:33:00+00:00    2024-01-17 18:34:00+00:00         8.469660e+01                                             2```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing 'trace' DataFrame loaded previously.
start_unix = 1705515929
end_unix = 1705517729

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using full series (before filtering)
grp_thresholds = trace.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].agg(
    global_p95=lambda s: float(np.nanpercentile(s.values, 95)) if len(s) > 0 else np.nan,
    global_p05=lambda s: float(np.nanpercentile(s.values, 5)) if len(s) > 0 else np.nan
)

# 2) Filter trace rows to the incident window (inclusive)
window = trace[(trace['timestamp'] >= start_unix) & (trace['timestamp'] <= end_unix)].copy()

# Merge thresholds into window rows
window = window.merge(grp_thresholds, on=['cmdb_id', 'trace_name'], how='left')

# Parse UTC timestamps for reporting
window['ts_utc'] = pd.to_datetime(window['timestamp'], unit='s', utc=True)

# 3) Mark anomalies:
# - if trace_name contains 'row_count' -> anomaly when value <= global_p05 (traffic drop)
# - else -> anomaly when value >= global_p95 (latency/error)
is_row_count = window['trace_name'].str.contains('row_count', case=False, na=False)
cond_row_count = is_row_count & (window['value'] <= window['global_p05'])
cond_other = (~is_row_count) & (window['value'] >= window['global_p95'])
window['anomaly'] = (cond_row_count | cond_other).fillna(False).astype(bool)

# 4) Aggregations per (cmdb_id, trace_name)
totals = window.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    total_points_in_window=('value', 'size'),
    anomaly_point_count_in_window=('anomaly', 'sum'),
    max_value_in_window=('value', 'max'),
    min_value_in_window=('value', 'min')
)

# Earliest and latest anomaly timestamps in UTC (for groups with anomalies)
earliest_latest = (
    window[window['anomaly']]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        earliest_anomaly_timestamp_utc=('ts_utc', 'min'),
        latest_anomaly_timestamp_utc=('ts_utc', 'max')
    )
)

# Longest consecutive anomaly streak in window per group
def longest_consecutive_streak(df_group):
    arr = df_group.sort_values('timestamp')['anomaly'].astype(bool).values
    max_run = 0
    cur = 0
    for val in arr:
        if val:
            cur += 1
            if cur > max_run:
                max_run = cur
        else:
            cur = 0
    return int(max_run)

streaks = (
    window.groupby(['cmdb_id', 'trace_name'])
    .apply(lambda g: longest_consecutive_streak(g))
    .reset_index(name='longest_consecutive_anomaly_streak_in_window')
)

# Assemble final summary starting from grp_thresholds (ensures every group has thresholds)
summary = grp_thresholds.merge(totals, on=['cmdb_id', 'trace_name'], how='left') \
                        .merge(earliest_latest, on=['cmdb_id', 'trace_name'], how='left') \
                        .merge(streaks, on=['cmdb_id', 'trace_name'], how='left')

# Fill missing numeric aggregates for groups absent in window
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['anomaly_point_count_in_window'] = summary['anomaly_point_count_in_window'].fillna(0).astype(int)
summary['longest_consecutive_anomaly_streak_in_window'] = summary['longest_consecutive_anomaly_streak_in_window'].fillna(0).astype(int)

# max/min in window remain NaN if no points
# earliest/latest anomaly timestamps remain NaT if no anomalies

# 5) Sort and limit to top 20 rows
summary_sorted = summary.sort_values(
    by=['anomaly_point_count_in_window', 'longest_consecutive_anomaly_streak_in_window'],
    ascending=[False, False]
)

final_cols = [
    'cmdb_id',
    'trace_name',
    'global_p95',
    'global_p05',
    'total_points_in_window',
    'anomaly_point_count_in_window',
    'earliest_anomaly_timestamp_utc',
    'latest_anomaly_timestamp_utc',
    'max_value_in_window',
    'min_value_in_window',
    'longest_consecutive_anomaly_streak_in_window'
]

trace_summary_top20 = summary_sorted[final_cols].head(20)

# Display the compact summary (variable output)
trace_summary_top20

```
Out[3]:
```
Summary of trace anomaly scan for the incident window (2024-01-17T18:25:29Z — 2024-01-17T18:55:29Z):

- What was done: global P95 and P05 were computed per (cmdb_id, trace_name) using the full series. Window rows were flagged as anomalies when:
  - trace_name contains "row_count" and value <= global_P05 (traffic drop), OR
  - otherwise value >= global_P95 (latency/error spike).

Key findings (compact)
- The top flagged groups are almost entirely error_rate traces across many services (checkoutservice, currencyservice, emailservice, frontendservice, etc.).
- For these error_rate groups:
  - global_p95 = 0.0 and global_p05 = 0.0
  - total_points_in_window = 25 and anomaly_point_count_in_window = 25 (i.e., all window points flagged)
  - max_value_in_window = 0.0 and min_value_in_window = 0.0
  - earliest anomaly = 2024-01-17T18:28:00Z, latest = 2024-01-17T18:52:00Z
  - longest consecutive anomaly streak = 25 (the whole window)
- Interpretation: these flags are caused by threshold equality (global P95 = 0) and the anomaly rule using >=. Since observed values are also 0, the rule marks them anomalous even though there is no observable non-zero error rate. This behavior likely produces false positives.
- There are no meaningful latency or row_count anomaly patterns shown in the top results — instead the dominant pattern is the error_rate = 0 case across many services.

Recommended next steps
1. Treat these error_rate flags as likely artifacts: verify whether error_rate = 0 is expected (healthy) or if metric collection/normalization produced zero values.
2. Adjust anomaly rule to avoid flagging when global_p95 equals global_p05 or when global_p95 == observed value (e.g., use value > global_p95, or require a margin/relative increase).
3. Correlate with other telemetry (metrics, logs) to confirm any real service issues (e.g., latency p95, row_count drops, socket/disk anomalies previously observed).
4. If you want, I can (a) list any trace groups where non-zero latency/error/larger deltas were flagged, or (b) show row_count traces to check for traffic drops.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                 trace_name  global_p95  global_p05  total_points_in_window  anomaly_point_count_in_window earliest_anomaly_timestamp_utc latest_anomaly_timestamp_utc  max_value_in_window  min_value_in_window  longest_consecutive_anomaly_streak_in_window
2   checkoutservice      trace.from_checkoutservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
6   checkoutservice      trace.from_frontendservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
10  checkoutservice                 trace.from_root.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
14  checkoutservice        trace.to_checkoutservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
18  checkoutservice        trace.to_currencyservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
22  checkoutservice           trace.to_emailservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
26  checkoutservice         trace.to_paymentservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
30  checkoutservice  trace.to_productcatalogservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
34  currencyservice      trace.from_checkoutservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
38  currencyservice      trace.from_frontendservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
42  currencyservice                 trace.from_root.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
46     emailservice      trace.from_checkoutservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
50     emailservice                 trace.from_root.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
54  frontendservice      trace.from_frontendservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
58  frontendservice                 trace.from_root.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
62  frontendservice        trace.to_checkoutservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
66  frontendservice        trace.to_currencyservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
70  frontendservice        trace.to_frontendservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
74  frontendservice  trace.to_productcatalogservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25
78  frontendservice  trace.to_recommendationservice.error_rate         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  0.0                  0.0                                            25```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing 'log' DataFrame. Define incident window (inclusive)
start_unix = 1705515929
end_unix = 1705517729

# 1) Compute global P95 and P05 per (cmdb_id, log_name) using full series
log_thresholds = log.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].agg(
    global_p95=lambda s: float(np.nanpercentile(s.values, 95)) if len(s) > 0 else np.nan,
    global_p05=lambda s: float(np.nanpercentile(s.values, 5)) if len(s) > 0 else np.nan
)

# 2) Filter log rows to the incident window (inclusive)
window = log[(log['timestamp'] >= start_unix) & (log['timestamp'] <= end_unix)].copy()

# Merge thresholds into window rows
window = window.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')

# Parse UTC timestamps for reporting
window['ts_utc'] = pd.to_datetime(window['timestamp'], unit='s', utc=True)

# 3) Flag anomalies:
is_row_count = window['log_name'].str.contains('row_count', case=False, na=False)
is_error_count = window['log_name'].str.contains('error_count', case=False, na=False)

cond_row_count = is_row_count & (window['value'] <= window['global_p05'])
cond_error_count = is_error_count & (window['value'] >= window['global_p95'])

window['anomaly'] = (cond_row_count | cond_error_count).fillna(False).astype(bool)

# 4) Aggregations per (cmdb_id, log_name)
totals = window.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    total_points_in_window=('value', 'size'),
    anomaly_point_count_in_window=('anomaly', 'sum'),
    min_value_in_window=('value', 'min'),
    max_value_in_window=('value', 'max')
)

# Earliest and latest anomaly timestamps in UTC (for groups with anomalies)
earliest_latest = (
    window[window['anomaly']]
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        earliest_anomaly_timestamp_utc=('ts_utc', 'min'),
        latest_anomaly_timestamp_utc=('ts_utc', 'max')
    )
)

# Longest consecutive anomaly streak in window per group
def longest_consecutive_streak(df_group):
    arr = df_group.sort_values('timestamp')['anomaly'].astype(bool).values
    max_run = 0
    cur = 0
    for val in arr:
        if val:
            cur += 1
            if cur > max_run:
                max_run = cur
        else:
            cur = 0
    return int(max_run)

streaks = (
    window.groupby(['cmdb_id', 'log_name'])
    .apply(lambda g: longest_consecutive_streak(g))
    .reset_index(name='longest_consecutive_anomaly_streak_in_window')
)

# Assemble final summary starting from log_thresholds to include all groups
summary = log_thresholds.merge(totals, on=['cmdb_id', 'log_name'], how='left') \
                        .merge(earliest_latest, on=['cmdb_id', 'log_name'], how='left') \
                        .merge(streaks, on=['cmdb_id', 'log_name'], how='left')

# Fill missing numeric aggregates for groups absent in window
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['anomaly_point_count_in_window'] = summary['anomaly_point_count_in_window'].fillna(0).astype(int)
summary['longest_consecutive_anomaly_streak_in_window'] = summary['longest_consecutive_anomaly_streak_in_window'].fillna(0).astype(int)

# Keep min/max as floats (NaN if no points). earliest/latest remain NaT if no anomalies.

# 5) Sort and limit to top 20 rows
summary_sorted = summary.sort_values(
    by=['anomaly_point_count_in_window', 'longest_consecutive_anomaly_streak_in_window'],
    ascending=[False, False]
)

final_cols = [
    'cmdb_id',
    'log_name',
    'global_p95',
    'global_p05',
    'total_points_in_window',
    'anomaly_point_count_in_window',
    'earliest_anomaly_timestamp_utc',
    'latest_anomaly_timestamp_utc',
    'min_value_in_window',
    'max_value_in_window',
    'longest_consecutive_anomaly_streak_in_window'
]

log_summary_top20 = summary_sorted[final_cols].head(20)

# Display the compact summary (variable output)
log_summary_top20

```
Out[4]:
```
Summary of log anomaly scan for the incident window (2024-01-17T18:25:29Z — 2024-01-17T18:55:29Z):

What I computed
- For each (cmdb_id, log_name) I computed global P95 and P05 over the entire series, filtered rows to the incident window, then flagged anomalies:
  - log.row_count: anomaly if value <= global_P05 (traffic drop)
  - log.error_count: anomaly if value >= global_P95 (error spike)

Key findings (compact)
- Widespread log.error_count flags:
  - Many services (adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, paymentservice, recommendationservice, shippingservice) have:
    - global_p95 = 0.0, total_points_in_window = 25, anomaly_point_count_in_window = 25
    - min/max in-window = 0, earliest anomaly 2024-01-17T18:28:00Z, latest 18:52:00Z, longest streak = 25
  - redis: log.error_count has 4/4 anomalies in-window (18:32 → 18:47), min/max = 0, streak = 4
  - Interpretation: these are flagged because global_p95 = 0 and observed values = 0, so the rule (>= global_p95) marks them anomalous — likely false positives or a threshold artifact rather than real error spikes.

- log.row_count traffic-drop flags (small, short drops):
  - Many services show 2 anomaly points in-window (out of 25): adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, paymentservice, recommendationservice, shippingservice.
  - Example values:
    - frontend log.row_count: global_p95=2795.6, global_p05=1518.6; in-window min=966, max=2818; anomaly_count=2 (earliest/latest anomaly span 18:28 → 18:52), longest streak = 1
    - currencyservice log.row_count: global_p95=2521.2, global_p05=1306.8; anomaly_count=2 (min=875, max=2537)
  - redis log.row_count: global_p95 = 5.0, global_p05 = 5.0; 4/4 points flagged (18:32 → 18:47), min/max = 5, streak = 4 (again a threshold-equality artifact likely)

Overall interpretation
- The log.error_count flags are dominated by global_p95 = 0 thresholds and observed zeros; these are likely false positives caused by the >= rule when the threshold is zero.
- The more actionable signals are the small number of log.row_count drops (2 points) in many services — these indicate brief traffic decreases during the window and merit investigation, though they are short (streak = 1) in most services.
- redis shows a consistent in-window pattern (4 points) for both error_count and row_count, but values are equal to thresholds (possible instrumentation/normalization issue).

Recommended next steps
1. Treat error_count=0 flags as likely artifacts — verify metric/log collection and consider changing anomaly rule (use > global_p95 or require positive values) to avoid flagging zeros.
2. Investigate the brief row_count drops (adservice, cartservice, frontend, currencyservice, etc.) around 18:28–18:52 — correlate with traces/metrics (traffic, latency, sockets/disk) to see if these drops correspond to user-visible impact.
3. Inspect redis telemetry and instrumentation for possible normalization issues (thresholds equal to observed values).
4. If you want, I can list numeric deltas or show the specific window rows for any of the flagged (cmdb_id, log_name) groups.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  global_p95  global_p05  total_points_in_window  anomaly_point_count_in_window earliest_anomaly_timestamp_utc latest_anomaly_timestamp_utc  min_value_in_window  max_value_in_window  longest_consecutive_anomaly_streak_in_window
0               adservice  log.error_count         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                    0                    0                                            25
2             cartservice  log.error_count         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                    0                    0                                            25
4         checkoutservice  log.error_count         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                    0                    0                                            25
6         currencyservice  log.error_count         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                    0                    0                                            25
8            emailservice  log.error_count         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                    0                    0                                            25
10               frontend  log.error_count         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                    0                    0                                            25
12         paymentservice  log.error_count         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                    0                    0                                            25
14  recommendationservice  log.error_count         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                    0                    0                                            25
18        shippingservice  log.error_count         0.0         0.0                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                    0                    0                                            25
16                  redis  log.error_count         0.0         0.0                       4                              4      2024-01-17 18:32:00+00:00    2024-01-17 18:47:00+00:00                    0                    0                                             4
17                  redis    log.row_count         5.0         5.0                       4                              4      2024-01-17 18:32:00+00:00    2024-01-17 18:47:00+00:00                    5                    5                                             4
1               adservice    log.row_count       468.8       258.4                      25                              2      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  163                  472                                             1
3             cartservice    log.row_count       815.2       456.8                      25                              2      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  274                  819                                             1
5         checkoutservice    log.row_count       103.2        44.6                      25                              2      2024-01-17 18:42:00+00:00    2024-01-17 18:52:00+00:00                   16                  105                                             1
7         currencyservice    log.row_count      2521.2      1306.8                      25                              2      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  875                 2537                                             1
9            emailservice    log.row_count        34.4        14.6                      25                              2      2024-01-17 18:42:00+00:00    2024-01-17 18:52:00+00:00                    5                   35                                             1
11               frontend    log.row_count      2795.6      1518.6                      25                              2      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  966                 2818                                             1
13         paymentservice    log.row_count        68.8        29.2                      25                              2      2024-01-17 18:42:00+00:00    2024-01-17 18:52:00+00:00                   10                   70                                             1
15  recommendationservice    log.row_count       587.6       340.4                      25                              2      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  204                  590                                             1
19        shippingservice    log.row_count       475.6       278.0                      25                              2      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00                  154                  498                                             1```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing 'trace' and 'grp_thresholds' if available; otherwise compute thresholds
try:
    grp_thresholds
except NameError:
    grp_thresholds = trace.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].agg(
        global_p95=lambda s: float(np.nanpercentile(s.values, 95)) if len(s) > 0 else np.nan,
        global_p05=lambda s: float(np.nanpercentile(s.values, 5)) if len(s) > 0 else np.nan
    )

# Target groups and patterns
target_cmdbs = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]
patterns = ['duration_mean','duration_p95','row_count','duration']  # 'duration' covers other duration variants

# Incident window (reuse existing if present)
try:
    start_unix, end_unix
except NameError:
    start_unix = 1705515929
    end_unix = 1705517729

# Filter thresholds to target groups and trace_name patterns
mask_thresholds = grp_thresholds['cmdb_id'].isin(target_cmdbs) & grp_thresholds['trace_name'].str.contains(
    '|'.join(patterns), case=False, na=False
)
grp_thresholds_filt = grp_thresholds[mask_thresholds].copy()

# Filter trace rows to target groups, patterns, and incident window
mask_window = (
    trace['cmdb_id'].isin(target_cmdbs) &
    trace['trace_name'].str.contains('|'.join(patterns), case=False, na=False) &
    (trace['timestamp'] >= start_unix) &
    (trace['timestamp'] <= end_unix)
)
window = trace[mask_window].copy()

# Merge thresholds into window rows
window = window.merge(grp_thresholds_filt, on=['cmdb_id', 'trace_name'], how='left')

# Parse UTC timestamps for reporting
window['ts_utc'] = pd.to_datetime(window['timestamp'], unit='s', utc=True)

# Determine anomaly conditions
is_row_count = window['trace_name'].str.contains('row_count', case=False, na=False)
cond_row_count = is_row_count & (window['value'] <= window['global_p05'])
cond_other = (~is_row_count) & (window['value'] >= window['global_p95'])
window['anomaly'] = (cond_row_count | cond_other).fillna(False).astype(bool)

# Aggregations per (cmdb_id, trace_name)
totals = window.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    total_points_in_window=('value', 'size'),
    anomaly_point_count_in_window=('anomaly', 'sum'),
    max_value_in_window=('value', 'max'),
    min_value_in_window=('value', 'min')
)

# Earliest and latest anomaly timestamps in UTC (for groups with anomalies)
earliest_latest = (
    window[window['anomaly']]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        earliest_anomaly_timestamp_utc=('ts_utc', 'min'),
        latest_anomaly_timestamp_utc=('ts_utc', 'max')
    )
)

# Longest consecutive anomaly streak in window per group
def longest_consecutive_streak(df_group):
    arr = df_group.sort_values('timestamp')['anomaly'].astype(bool).values
    max_run = 0
    cur = 0
    for val in arr:
        if val:
            cur += 1
            if cur > max_run:
                max_run = cur
        else:
            cur = 0
    return int(max_run)

streaks = (
    window.groupby(['cmdb_id', 'trace_name'])
    .apply(lambda g: longest_consecutive_streak(g))
    .reset_index(name='longest_consecutive_anomaly_streak_in_window')
)

# Assemble final summary starting from filtered thresholds to include all target groups
summary = grp_thresholds_filt.merge(totals, on=['cmdb_id', 'trace_name'], how='left') \
                            .merge(earliest_latest, on=['cmdb_id', 'trace_name'], how='left') \
                            .merge(streaks, on=['cmdb_id', 'trace_name'], how='left')

# Fill missing numeric aggregates for groups absent in window
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['anomaly_point_count_in_window'] = summary['anomaly_point_count_in_window'].fillna(0).astype(int)
summary['longest_consecutive_anomaly_streak_in_window'] = summary['longest_consecutive_anomaly_streak_in_window'].fillna(0).astype(int)

# Keep max/min as floats (NaN if no points). earliest/latest remain NaT if no anomalies.

# Sort by anomaly count desc, then streak desc, limit to top 20
summary_sorted = summary.sort_values(
    by=['anomaly_point_count_in_window', 'longest_consecutive_anomaly_streak_in_window'],
    ascending=[False, False]
)

final_cols = [
    'cmdb_id',
    'trace_name',
    'global_p95',
    'global_p05',
    'total_points_in_window',
    'anomaly_point_count_in_window',
    'earliest_anomaly_timestamp_utc',
    'latest_anomaly_timestamp_utc',
    'max_value_in_window',
    'min_value_in_window',
    'longest_consecutive_anomaly_streak_in_window'
]

trace_summary_top20 = summary_sorted[final_cols].head(20)

# Display the compact summary (variable output)
trace_summary_top20

```
Out[5]:
```
Summary of trace analysis for selected services during the incident window (2024-01-17T18:25:29Z — 2024-01-17T18:55:29Z)

What I did (brief)
- Computed global P95 and P05 for each (cmdb_id, trace_name) using the full trace series (before any window filtering).
- Filtered trace rows to the incident window and flagged anomalies:
  - row_count → anomaly if value <= global_P05 (traffic drop)
  - other traces (duration*, duration_p95, duration_mean) → anomaly if value >= global_P95 (latency/error spike)
- Produced a compact per-group summary limited to the top results.

Top findings (compact)
- productcatalogservice — trace.from_frontendservice.duration_p95
  - global_p95 ≈ 3.4e-05, global_p05 ≈ 2.1e-05
  - 25 points in-window, 5 anomaly points (earliest 2024-01-17T18:43:00Z, latest 18:52:00Z)
  - max in-window ≈ 3.5e-05, min ≈ 2.0e-05, longest consecutive anomaly streak = 3
  - Interpretation: measurable latency spikes in productcatalogservice for requests from the frontend starting ~18:43 UTC.

- checkoutservice / currencyservice — row_count between services
  - checkoutservice trace.to_currencyservice.row_count and currencyservice trace.from_checkoutservice.row_count:
    - global_p95=120.6, global_p05=51.0
    - 25 points in-window, 3 anomaly points each (earliest 18:41:00Z → latest 18:52:00Z)
    - max/min in-window: 135 / 25 (shows drop down to 25)
    - longest streak = 2
  - Interpretation: brief traffic drops on the checkout ↔ currency interaction starting ~18:41 UTC.

- emailservice — trace.from_root.duration_mean
  - global_p95 ≈ 5.2e-05, global_p05 ≈ 4.1e-05
  - 25 points in-window, 3 anomaly points (18:28 → 18:40 UTC), max ~6.3e-05
  - Interpretation: brief early latency increases for emailservice traces.

- productcatalogservice / recommendationservice latency links
  - trace.from_recommendationservice.duration_p95 (productcatalogservice) and recommendationservice trace.to_productcatalogservice.duration_p95:
    - each 25 points in-window, 3 anomaly points (18:44 → 18:52 UTC), small increases above p95
    - Interpretation: latency between recommendationservice and productcatalogservice increased in the later part of the window.

- Other checkoutservice traces (duration_mean, duration_p95, row_count, to_checkoutservice.*)
  - Many checkout traces show 2 anomaly points each (various time ranges mostly after ~18:42–18:49).
  - Interpretation: checkout service shows short-lived latency and traffic anomalies concentrated in the latter half of the window.

Overall interpretation
- Two classes of meaningful signals:
  1. Latency spikes (duration_p95 / duration_mean) observed notably for productcatalogservice (from frontend) around 18:43–18:52 UTC and for some recommendationservice ↔ productcatalogservice interactions.
  2. Short traffic drops (row_count) on checkout ↔ currency flows starting ~18:41 UTC (3 anomaly points, brief streaks).
- Timing: emailservice latency anomalies occurred earlier (~18:28–18:40); productcatalogservice and recommendationservice latency issues cluster later (~18:43–18:52). Checkout ↔ currency traffic drops begin ~18:41.

Recommended next steps
- Investigate productcatalogservice: correlate the frontend → productcatalog latency spikes with service metrics (CPU, mem, diskio), socket counts, and logs for that host.
- Investigate communication between recommendationservice and productcatalogservice (latency increase).
- Investigate checkout ↔ currency traffic drops: check load balancer, network path, and upstream service (currencyservice) health at ~18:41–18:52 UTC.
- Correlate these trace anomalies with the previously observed metric-level socket/diskio signals and with log row_count drops to determine if the issue is networking, I/O, or a service-side latency problem.

If you want, I can:
- Show the numeric global_p95/global_p05 and the window rows for any specific group above, or
- Correlate these groups with metric.csv entries (socket, diskio, mem) in the same time window.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                     trace_name  global_p95  global_p05  total_points_in_window  anomaly_point_count_in_window earliest_anomaly_timestamp_utc latest_anomaly_timestamp_utc  max_value_in_window  min_value_in_window  longest_consecutive_anomaly_streak_in_window
43  productcatalogservice        trace.from_frontendservice.duration_p95    0.000034    0.000021                      25                              5      2024-01-17 18:43:00+00:00    2024-01-17 18:52:00+00:00             0.000035             0.000020                                             3
14        checkoutservice             trace.to_currencyservice.row_count  120.600000   51.000000                      25                              3      2024-01-17 18:41:00+00:00    2024-01-17 18:52:00+00:00           135.000000            25.000000                                             2
26        currencyservice           trace.from_checkoutservice.row_count  120.600000   51.000000                      25                              3      2024-01-17 18:41:00+00:00    2024-01-17 18:52:00+00:00           135.000000            25.000000                                             2
36           emailservice                  trace.from_root.duration_mean    0.000052    0.000041                      25                              3      2024-01-17 18:28:00+00:00    2024-01-17 18:40:00+00:00             0.000063             0.000040                                             1
46  productcatalogservice  trace.from_recommendationservice.duration_p95    0.000035    0.000023                      25                              3      2024-01-17 18:44:00+00:00    2024-01-17 18:52:00+00:00             0.000038             0.000022                                             1
61  recommendationservice    trace.to_productcatalogservice.duration_p95    0.000035    0.000023                      25                              3      2024-01-17 18:44:00+00:00    2024-01-17 18:52:00+00:00             0.000038             0.000022                                             1
42  productcatalogservice       trace.from_frontendservice.duration_mean    0.000018    0.000013                      25                              2      2024-01-17 18:45:00+00:00    2024-01-17 18:46:00+00:00             0.000020             0.000012                                             2
0         checkoutservice       trace.from_checkoutservice.duration_mean    0.070861    0.003630                      25                              2      2024-01-17 18:43:00+00:00    2024-01-17 18:52:00+00:00             0.084692             0.003595                                             1
1         checkoutservice        trace.from_checkoutservice.duration_p95    0.203600    0.005456                      25                              2      2024-01-17 18:49:00+00:00    2024-01-17 18:52:00+00:00             0.203847             0.005382                                             1
2         checkoutservice           trace.from_checkoutservice.row_count  292.600000  125.600000                      25                              2      2024-01-17 18:42:00+00:00    2024-01-17 18:52:00+00:00           310.000000            52.000000                                             1
3         checkoutservice       trace.from_frontendservice.duration_mean    0.658550    0.081458                      25                              2      2024-01-17 18:43:00+00:00    2024-01-17 18:52:00+00:00             0.769472             0.060176                                             1
4         checkoutservice        trace.from_frontendservice.duration_p95    1.232080    0.123787                      25                              2      2024-01-17 18:44:00+00:00    2024-01-17 18:52:00+00:00             1.401499             0.103738                                             1
5         checkoutservice           trace.from_frontendservice.row_count   34.400000   15.400000                      25                              2      2024-01-17 18:42:00+00:00    2024-01-17 18:52:00+00:00            35.000000             6.000000                                             1
6         checkoutservice                  trace.from_root.duration_mean    0.013851    0.004218                      25                              2      2024-01-17 18:32:00+00:00    2024-01-17 18:38:00+00:00             0.014577             0.003989                                             1
7         checkoutservice                   trace.from_root.duration_p95    0.068436    0.006408                      25                              2      2024-01-17 18:32:00+00:00    2024-01-17 18:38:00+00:00             0.070451             0.006242                                             1
8         checkoutservice                      trace.from_root.row_count  144.600000   71.800000                      25                              2      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00           159.000000            37.000000                                             1
9         checkoutservice         trace.to_checkoutservice.duration_mean    0.070861    0.003630                      25                              2      2024-01-17 18:43:00+00:00    2024-01-17 18:52:00+00:00             0.084692             0.003595                                             1
10        checkoutservice          trace.to_checkoutservice.duration_p95    0.203600    0.005456                      25                              2      2024-01-17 18:49:00+00:00    2024-01-17 18:52:00+00:00             0.203847             0.005382                                             1
11        checkoutservice             trace.to_checkoutservice.row_count  292.600000  125.600000                      25                              2      2024-01-17 18:42:00+00:00    2024-01-17 18:52:00+00:00           310.000000            52.000000                                             1
12        checkoutservice         trace.to_currencyservice.duration_mean    0.001083    0.000139                      25                              2      2024-01-17 18:28:00+00:00    2024-01-17 18:31:00+00:00             0.001731             0.000136                                             1```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing 'metric' DataFrame loaded earlier.

cmdb_target = 'productcatalogservice'
start_unix = 1705515929
end_unix = 1705517729

# 1) Compute global P95 per kpi_name for productcatalogservice using the full series
metric_pc_full = metric[metric['cmdb_id'] == cmdb_target]
global_p95_df = (
    metric_pc_full
    .groupby('kpi_name', as_index=False)['value']
    .agg(global_p95=lambda s: float(np.nanpercentile(s.values, 95)) if len(s) > 0 else np.nan)
)
global_p95_df['cmdb_id'] = cmdb_target

# 2) Filter metric rows to the incident window (inclusive) for productcatalogservice
window = metric_pc_full[(metric_pc_full['timestamp'] >= start_unix) & (metric_pc_full['timestamp'] <= end_unix)].copy()
window['ts_utc'] = pd.to_datetime(window['timestamp'], unit='s', utc=True)

# 3) Merge global_p95 into window rows and mark anomalies (value >= global_p95)
window = window.merge(global_p95_df[['kpi_name', 'global_p95']], on='kpi_name', how='left')
window['anomaly'] = (window['value'] >= window['global_p95']).fillna(False).astype(bool)

# Helper: longest consecutive anomaly streak for a group's window rows
def longest_streak(group):
    arr = group.sort_values('timestamp')['anomaly'].astype(bool).values
    max_run = 0
    cur = 0
    for v in arr:
        if v:
            cur += 1
            if cur > max_run:
                max_run = cur
        else:
            cur = 0
    return int(max_run)

# Aggregations per kpi_name
agg_window = window.groupby('kpi_name', as_index=False).agg(
    total_points_in_window=('value', 'size'),
    anomaly_point_count_in_window=('anomaly', 'sum'),
    min_value_in_window=('value', 'min'),
    mean_value_in_window=('value', 'mean'),
    max_value_in_window=('value', 'max')
)

# Earliest/latest anomaly timestamps per kpi_name
earliest_latest = (
    window[window['anomaly']]
    .groupby('kpi_name', as_index=False)
    .agg(
        earliest_anomaly_timestamp_utc=('ts_utc', 'min'),
        latest_anomaly_timestamp_utc=('ts_utc', 'max')
    )
)

# Longest consecutive streaks
streaks = (
    window.groupby('kpi_name')
    .apply(lambda g: longest_streak(g))
    .reset_index(name='longest_consecutive_anomaly_streak_in_window')
)

# 4) Assemble final summary: start from global_p95_df to include all kpi_names
summary = global_p95_df.merge(agg_window, on='kpi_name', how='left') \
                       .merge(earliest_latest, on='kpi_name', how='left') \
                       .merge(streaks, on='kpi_name', how='left')

# Fill missing numeric aggregates for kpis absent in the window
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['anomaly_point_count_in_window'] = summary['anomaly_point_count_in_window'].fillna(0).astype(int)
summary['min_value_in_window'] = summary['min_value_in_window'].astype(float)
summary['mean_value_in_window'] = summary['mean_value_in_window'].astype(float)
summary['max_value_in_window'] = summary['max_value_in_window'].astype(float)
summary['longest_consecutive_anomaly_streak_in_window'] = summary['longest_consecutive_anomaly_streak_in_window'].fillna(0).astype(int)

# Ensure earliest/latest anomaly timestamps are proper dtype (remain NaT if none)
# 5) Sort and limit to top 20 rows
summary_sorted = summary.sort_values(
    by=['anomaly_point_count_in_window', 'longest_consecutive_anomaly_streak_in_window'],
    ascending=[False, False]
)

final_cols = [
    'cmdb_id',
    'kpi_name',
    'global_p95',
    'total_points_in_window',
    'anomaly_point_count_in_window',
    'earliest_anomaly_timestamp_utc',
    'latest_anomaly_timestamp_utc',
    'min_value_in_window',
    'mean_value_in_window',
    'max_value_in_window',
    'longest_consecutive_anomaly_streak_in_window'
]

summary_top20 = summary_sorted[final_cols].head(20)

# Display the compact summary table
summary_top20

```
Out[6]:
```
Summary for productcatalogservice metrics during the incident window (2024-01-17T18:25:29Z — 2024-01-17T18:55:29Z):

What was computed
- Global P95 per kpi_name was computed using the full series (no window filtering).
- Window rows were flagged as anomalies when value >= that global P95.
- The table below is a compact summary per kpi_name (only top results present in data).

Key findings (compact)
- socket
  - global_p95 = 4.0
  - total points in window = 25, anomaly points = 25
  - earliest = 2024-01-17T18:28:00Z, latest = 2024-01-17T18:52:00Z
  - min = mean = max = 4.0, longest consecutive anomaly streak = 25
  - Interpretation: socket is constant at the P95 value across the whole window — likely a threshold/equality artifact or stable metric, not a transient spike.

- latency-50
  - global_p95 ≈ 0.0699
  - 25 points in window, 2 anomaly points (18:51 → 18:52 UTC)
  - min ≈ 0.0017, mean ≈ 0.0260, max ≈ 0.1005
  - Short-lived latency elevations near the end of the window.

- cpu
  - global_p95 ≈ 3.553
  - 25 points, 2 anomaly points (18:30 → 18:34 UTC)
  - min ≈ 3.0926, mean ≈ 3.3471, max ≈ 3.6184
  - Brief CPU values above P95 earlier in the window.

- latency-90
  - global_p95 ≈ 0.2201
  - 25 points, 2 anomaly points (18:49 → 18:52 UTC)
  - min ≈ 0.00435, mean ≈ 0.10899, max ≈ 0.22024
  - Short-lived p90 latency increases toward the end of the window.

- mem
  - global_p95 ≈ 1.0735e7
  - 25 points, 2 anomaly points (18:44 → 18:48 UTC)
  - min ≈ 1.0121e7, mean ≈ 1.04997e7, max ≈ 1.08463e7
  - Small memory increases for a short interval mid-window.

- workload
  - global_p95 ≈ 154.10
  - 25 points, 2 anomaly points (18:28 → 18:34 UTC)
  - min ≈ 103.34, mean ≈ 128.76, max ≈ 157.13
  - Brief elevated workload early in the window.

Overall interpretation and recommended next steps
- The only sustained, full-window flags are on socket (25/25) — but because socket values equal the computed P95 exactly (min = max = P95), this likely reflects threshold/equality behavior or a constant instrumentation value rather than a clear operational degradation.
- Other KPIs (latency-50, latency-90, cpu, mem, workload) show short-lived anomalies (typically 1–2 points) at different times within the window:
  - CPU and workload spikes early (~18:30–18:34).
  - Memory bump mid-window (~18:44–18:48).
  - Latency spikes late (~18:49–18:52).
- Correlate these brief metric anomalies with trace and log signals for productcatalogservice (and services calling it) to determine impact:
  - Check the trace latency anomalies previously observed for productcatalogservice (frontend→productcatalog) around ~18:43–18:52.
  - Verify whether socket constant flags are instrumentation/normalization artifacts and adjust anomaly rules (e.g., use > global_p95 or require delta) to avoid equality-triggered false positives.

If you want, I can show the raw window rows for any specific kpi_name above or correlate these timestamps with trace/log anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name    global_p95  total_points_in_window  anomaly_point_count_in_window earliest_anomaly_timestamp_utc latest_anomaly_timestamp_utc  min_value_in_window  mean_value_in_window  max_value_in_window  longest_consecutive_anomaly_streak_in_window
4  productcatalogservice      socket  4.000000e+00                      25                             25      2024-01-17 18:28:00+00:00    2024-01-17 18:52:00+00:00         4.000000e+00          4.000000e+00         4.000000e+00                                            25
1  productcatalogservice  latency-50  6.991141e-02                      25                              2      2024-01-17 18:51:00+00:00    2024-01-17 18:52:00+00:00         1.704734e-03          2.603075e-02         1.005459e-01                                             2
0  productcatalogservice         cpu  3.553004e+00                      25                              2      2024-01-17 18:30:00+00:00    2024-01-17 18:34:00+00:00         3.092560e+00          3.347130e+00         3.618395e+00                                             1
2  productcatalogservice  latency-90  2.201013e-01                      25                              2      2024-01-17 18:49:00+00:00    2024-01-17 18:52:00+00:00         4.350715e-03          1.089865e-01         2.202368e-01                                             1
3  productcatalogservice         mem  1.073471e+07                      25                              2      2024-01-17 18:44:00+00:00    2024-01-17 18:48:00+00:00         1.012074e+07          1.049972e+07         1.084628e+07                                             1
5  productcatalogservice    workload  1.540968e+02                      25                              2      2024-01-17 18:28:00+00:00    2024-01-17 18:34:00+00:00         1.033355e+02          1.287603e+02         1.571333e+02                                             1```
```